# Gaussian Mixture Model Anomaly Detection for TLS Profiling

This notebook evaluates one-class Gaussian Mixture Model baselines on extracted TLS traffic features. For each target label, the detector is trained only on flows from that label and then evaluated on its ability to separate that label from the rest of the traffic using negative log-likelihood as the anomaly score.


In [ ]:
import sys
from pathlib import Path
sys.path.append('../../src')

### PARAMETERS:

In [2]:
train_features_path = ["../data-ext/malware_train.parquet", "../data-ext/apps_train.parquet"]
val_features_path = ["../data-ext/malware_val.parquet", "../data-ext/apps_val.parquet"]
test_features_path = ["../data-ext/malware_test.parquet", "../data-ext/apps_test.parquet"]
train_labels_path = ["../data-ext/malware_train_labels.parquet", "../data-ext/apps_train_labels.parquet"]
val_labels_path = ["../data-ext/malware_val_labels.parquet", "../data-ext/apps_val_labels.parquet"]
test_labels_path = ["../data-ext/malware_test_labels.parquet", "../data-ext/apps_test_labels.parquet"]

category_labels = ["system", "malware", "application"]


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families carry the most discriminative signal for the GMM baseline.


In [3]:
feature_groups = {
    "FLOW": ["bs", "ps", "br", "pr", "td"],
    "CTLS": ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"],
    "STLS": ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"],
    "REC": ["tls.rec"],
}

FLOW = feature_groups["FLOW"]
CTLS = feature_groups["CTLS"]
STLS = feature_groups["STLS"]
REC = feature_groups["REC"]

In [4]:
import pandas as pd
from tls_profiling.preprocessing import build_and_fit_preprocessor

def read_parquet_files(paths):
    if isinstance(paths, (str, Path)):
        paths = [paths]
    return pd.concat([pd.read_parquet(Path(path)) for path in paths], ignore_index=True)

print("Loading extracted features from parameterized parquet paths.")
df_train = read_parquet_files(train_features_path)
df_val = read_parquet_files(val_features_path)
df_test = read_parquet_files(test_features_path)
df_train_label = read_parquet_files(train_labels_path)
df_val_label = read_parquet_files(val_labels_path)
df_test_label = read_parquet_files(test_labels_path)

preprocessor = build_and_fit_preprocessor(df_train)
print("Built preprocessor from df_train.")


ModuleNotFoundError: No module named 'tls_profiling'

In [5]:
from tls_profiling.baselines.model_gmm import GaussianMixtureDetector, Config
import numpy as np

GMM_CONFIG = Config(
    n_components=5,
    covariance_type="diag",
    reg_covar=1e-5,
    max_iter=200,
    n_init=2,
    max_train_samples=20_000,
)

def train_detector(train: np.ndarray) -> GaussianMixtureDetector:
    detector = GaussianMixtureDetector(GMM_CONFIG)
    detector.fit(train)
    return detector

from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"unknown class\(es\) .* will be ignored",
    category=UserWarning,
    module=r"sklearn\.preprocessing\._label",
)

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def choose_f1_threshold(y_true, anomaly_score):
    precision, recall, thresholds = precision_recall_curve(y_true, anomaly_score)

    if len(thresholds) == 0:
        return float("inf")

    f1_scores = (2 * precision[:-1] * recall[:-1]) / np.clip(
        precision[:-1] + recall[:-1],
        a_min=1e-12,
        a_max=None,
    )
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])

def evaluate(feature_set, normal_label):
    # x_train: only normal traffic
    x_train = df_train.loc[df_train_label["category"] == normal_label].reset_index(drop=True)
    # x_val: mixed traffic used to tune the F1 threshold
    x_val = df_val
    y_val = (df_val_label["category"] != normal_label).astype(int)
    # y_test: 1 = anomaly, 0 = normal
    y_test = (df_test_label["category"] != normal_label).astype(int)
    x_test = df_test

    # prepare datasets
    x_train_transformed = select_feature_set(preprocessor.transform(x_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(x_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(x_test), feature_set)

    # create detector on TRAIN
    detector = train_detector(x_train_transformed)

    # choose the F1 threshold on the mixed validation split
    val_scores = detector.score(x_val_transformed)
    threshold = choose_f1_threshold(y_val, val_scores)

    # evaluate on TEST
    anomaly_score = detector.score(x_test_transformed)

    return y_test, anomaly_score, threshold

def debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score):
    """
    Write the intermediate data to CSV file.
    """
    feature_set_name = "_".join(feature_set)
    class_label_name = normal_label

    output_path = f"tmp/gmm_{class_label_name}_{feature_set_name}.csv"
    pd.DataFrame({
        "y_test": y_test,
        "y_pred": y_pred,
        "anomaly_score": anomaly_score,
    }).to_csv(output_path, index=False)

def compute_scores(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)
    auc = roc_auc_score(y_test, anomaly_score)
    ap = average_precision_score(y_test, anomaly_score)
    y_pred = (anomaly_score >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)

    debug_csv(feature_set, normal_label, y_test, y_pred, anomaly_score)
    return {"auc_score": auc, "ap_score": ap, "f1_score": f1}

def plot_curves(feature_set, normal_label):
    y_test, anomaly_score, threshold = evaluate(feature_set, normal_label)

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    PrecisionRecallDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="GMM PR-AUC",
        ax=axes[0],
    )
    axes[0].set_title(f"{normal_label} Precision-Recall")

    RocCurveDisplay.from_predictions(
        y_test,
        anomaly_score,
        name="GMM AUC-ROC",
        ax=axes[1],
    )
    axes[1].set_title(f"{normal_label} ROC Curve")

    plt.tight_layout()
    plt.show()


## Evaluation

For each label in `system`, `unknown`, and `malware`, that label is treated as the in-profile or normal class. All remaining labels are treated as anomalies.

The evaluation uses three disjoint splits:

- `train`: only samples from the selected normal class, used to fit the GMM detector
- `validation`: mixed traffic, used to choose the anomaly-score threshold that maximizes `F1`
- `test`: held-out mixed traffic, used only for final reporting

Feature ablation:

- each feature group is evaluated individually
- all pairwise, triple, and full combinations are also evaluated

Reported metrics:

- `AUC-ROC`: how well the negative log-likelihood ranks the selected class against the rest over all thresholds
- `AP`: average precision, which is especially useful when the class balance is uneven
- `F1`: test-set score obtained with the threshold selected on the validation split

Practical notes for this baseline:

- the detector uses a diagonal-covariance GMM with `5` components and small covariance regularization
- each fit is capped at `20,000` training samples to keep the full ablation study tractable on this dataset while preserving the same evaluation protocol as the PCA and Isolation Forest notebooks


In [6]:
from itertools import combinations

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)

        for label in category_labels:
            display(f"Scoring {label}@{selected_features}...")
            scores = compute_scores(selected_features, label)

            rows.append({
                "FeatureSet": feature_set_name,
                "ClassLabel": label,
                "AucScore": scores["auc_score"],
                "ApScore": scores["ap_score"],
                "F1Score": scores["f1_score"],
            })

results_df = pd.DataFrame(rows, columns=["FeatureSet", "ClassLabel", "AucScore", "ApScore", "F1Score"])
display(results_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
0,FLOW,system,0.883791,0.918384,0.869529
1,FLOW,malware,0.511965,0.485483,0.698350
2,CTLS,system,0.915659,0.939960,0.916964
3,CTLS,malware,0.875400,0.913846,0.849544
4,STLS,system,0.978204,0.982723,0.974372
5,STLS,malware,0.873462,0.880112,0.862298
6,REC,system,0.975920,0.981615,0.965681
7,REC,malware,0.823500,0.862831,0.746112
8,"FLOW, CTLS",system,0.932826,0.961888,0.941254
9,"FLOW, CTLS",malware,0.863326,0.898920,0.845583


## System Profile

The table below reports the GMM baseline for the `system` profile across all feature-group combinations. Because the anomaly score is density-based rather than reconstruction-based, strong results here indicate that system traffic occupies a compact and repeatable region of the feature space under the fitted mixture model.


In [7]:
system_df = results_df[results_df["ClassLabel"] == "system"].sort_values("F1Score", ascending=False)
display(system_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
4,STLS,system,0.978204,0.982723,0.974372
20,"FLOW, CTLS, STLS",system,0.987732,0.993287,0.972578
14,"CTLS, STLS",system,0.988070,0.992837,0.972508
28,"FLOW, CTLS, STLS, REC",system,0.978713,0.989542,0.972201
26,"CTLS, STLS, REC",system,0.978703,0.989547,0.972144
24,"FLOW, STLS, REC",system,0.981346,0.986504,0.970697
18,"STLS, REC",system,0.982492,0.987624,0.970634
10,"FLOW, STLS",system,0.963410,0.972141,0.969755
6,REC,system,0.975920,0.981615,0.965681
12,"FLOW, REC",system,0.967149,0.977270,0.960348


## Malware Profile

This section isolates the GMM results for the `malware` profile. Compared with Isolation Forest or PCA, the GMM baseline can be especially informative when malware traffic is multi-modal and several behavior clusters need to be modeled as normal within the same one-class experiment.


In [8]:
malware_df = results_df[results_df["ClassLabel"] == "malware"].sort_values("F1Score", ascending=False)
display(malware_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
19,"STLS, REC",malware,0.926930,0.922940,0.900401
21,"FLOW, CTLS, STLS",malware,0.900465,0.925204,0.893076
25,"FLOW, STLS, REC",malware,0.905178,0.885592,0.877877
11,"FLOW, STLS",malware,0.907734,0.901865,0.873184
27,"CTLS, STLS, REC",malware,0.898009,0.880861,0.866730
17,"CTLS, REC",malware,0.907356,0.925061,0.866589
29,"FLOW, CTLS, STLS, REC",malware,0.897915,0.880242,0.865582
5,STLS,malware,0.873462,0.880112,0.862298
23,"FLOW, CTLS, REC",malware,0.904753,0.922968,0.860907
15,"CTLS, STLS",malware,0.860609,0.870525,0.854631


## Application Profile

This section isolates the GMM results for the `application` profile. Compared with Isolation Forest or PCA, the GMM baseline can be especially informative when application traffic is multi-modal and several behavior clusters need to be modeled as normal within the same one-class experiment.

In [9]:
application_df = results_df[results_df["ClassLabel"] == "application"].sort_values("F1Score", ascending=False)
display(application_df)


,FeatureSet,ClassLabel,AucScore,ApScore,F1Score
